### This notebook is for filtering objects we collected and getting their periods so that they can be evaluated using periodogram and light curve analysis

**Many parts of the code below are modified according to how we want to filter the data we're working with**

In [2]:
from astroquery.gaia import Gaia
from astroquery.vizier import Vizier
from astropy.coordinates import SkyCoord
import astropy.units as u
import csv
import pandas as pd
from lsst.rsp import get_tap_service

from sklearn import svm

Get current working directory via os so that the code can be used by anyone

In [36]:
path = os.getcwd()
# catalog_path = os.path.join(path, 'matched_diaobj_merged.csv')
catalog_path = os.path.join(path, 'predictions.csv') #change this path as needed
catalog_path
df = pd.read_csv(catalog_path)

In [37]:
pd.reset_option('display.max_rows')

We only want to work with objects predicted to be RR Lyraes (for now) because they will have a more defined light curve with the short observation period

In [38]:
df = df[(df['Predictions'] == 'RRLyrae') & (df['Class'] == 'Unlabeled')]

Filter out all objects that don't have at least 10 observations in one of the three main bands we're analyzing (g, r, i)

In [39]:
# df = df[(df['g_psfFluxNdata'] >= 10) | (df['r_psfFluxNdata'] >= 10) | (df['i_psfFluxNdata'] > 10)]
df = df[(df['r_psfFluxNdata'] >= 10) | (df['i_psfFluxNdata'] >= 10)]

In [40]:
df

,diaObjectId,ra,dec,nDiaSources,i_psfFluxNdata,r_psfFluxNdata,r_psfFluxStetsonJ,i_psfFluxStetsonJ,r_psfFluxChi2,i_psfFluxChi2,r_psfFluxChi2_red,i_psfFluxChi2_red,Class,g_psfFluxMean,i_psfFluxMean,r_psfFluxMean,g_psfFluxErr,i_psfFluxErr,r_psfFluxErr,g_weighted_mean_mag,i_weighted_mean_mag,r_weighted_mean_mag,g_fluxMagMin,i_fluxMagMin,r_fluxMagMin,g_fluxMagMax,i_fluxMagMax,r_fluxMagMax,g_fluxMagPerc05,i_fluxMagPerc05,r_fluxMagPerc05,g_fluxMagPerc95,i_fluxMagPerc95,r_fluxMagPerc95,g_fluxMagMedian,i_fluxMagMedian,r_fluxMagMedian,g_psfFluxMedian,i_psfFluxMedian,r_psfFluxMedian,g_fluxMagMaxSlope,i_fluxMagMaxSlope,r_fluxMagMaxSlope,g_psfFluxSigma,i_psfFluxSigma,r_psfFluxSigma,g_psfFluxMAD,i_psfFluxMAD,r_psfFluxMAD,g_fluxMagSkew,i_fluxMagSkew,r_fluxMagSkew,g_r,r_i,g_i,g_amp,i_amp,r_amp,g_psfFluxSigma_norm,i_psfFluxSigma_norm,r_psfFluxSigma_norm,g_psfFluxMAD_norm,i_psfFluxMAD_norm,r_psfFluxMAD_norm,Predictions
6,604062686048682065,40.058354,-34.669641,17,10.0,7.0,13.782720,7.641371,1509.970,743.383,251.660889,82.598165,Unlabeled,13385.049805,19409.099609,20331.078125,158.539001,244.567734,175.794739,21.083750,20.685913,20.618013,21.002224,20.558735,20.478302,21.177109,20.863914,20.963402,21.006704,20.568144,20.493887,21.170298,20.799638,20.874889,21.081898,20.687975,20.602322,13418.300293,19266.800781,20848.300781,351.681086,350.802619,343.919391,1010.573529,1540.891892,2310.213759,810.799805,822.800781,1036.000000,0.144058,0.464276,1.167591,0.465736,-0.067900,0.397837,0.163594,0.231494,0.381002,0.075500,0.079390,0.113630,0.060425,0.042706,0.049692,RRLyrae
7,604062686048682098,40.120221,-34.784352,16,4.0,12.0,4.581863,3.748847,731.713,124.485,66.519376,41.495008,Unlabeled,7702.240234,11517.312500,10823.839844,152.697998,224.611160,161.242783,21.683405,21.210251,21.301123,21.682003,21.115099,21.047628,21.684441,21.409576,21.452473,21.682196,21.129737,21.070629,21.684389,21.400403,21.451560,21.683929,21.180405,21.356697,7698.890137,12241.599609,10406.950195,4.653597,94.439072,155.460180,9.118822,1267.269619,1199.016929,3.620117,758.900391,682.250488,-1.429044,0.337751,-1.269174,0.382282,0.090872,0.473154,0.002194,0.270666,0.380931,0.001184,0.110032,0.110776,0.000470,0.061994,0.065557,RRLyrae
8,604062686048682133,40.093003,-34.629648,16,4.0,12.0,4.733237,8.266910,461.790,292.839,41.980924,97.613007,Unlabeled,50388.800781,181444.265625,129995.437500,227.648331,488.071991,363.965942,19.650953,18.253801,18.620802,19.550991,18.154505,18.164358,19.831341,18.295330,18.724960,19.553089,18.194359,18.551682,19.805404,18.294205,18.686539,19.571974,18.265263,18.642912,53853.000000,179430.000000,126717.000000,628.988822,216.408381,777.134497,6930.224528,6865.561071,15224.258336,1050.898438,4539.000000,2606.000000,1.696935,-1.539695,-3.875892,1.030151,0.367001,1.397152,0.252315,0.099846,0.134857,0.137535,0.037838,0.117114,0.019514,0.025297,0.020566,RRLyrae
20,604062754768158762,39.946244,-34.640705,28,11.0,17.0,44.931238,33.638724,94982.700,20606.500,5936.421387,2060.653516,Unlabeled,19655.250000,45467.070312,63491.824219,165.864990,304.264618,265.667542,20.645439,19.629459,19.325100,20.371498,19.352684,18.979301,21.072275,20.151840,20.224380,20.406537,19.356421,19.096092,21.037236,20.105694,20.157889,20.721886,20.037510,19.282398,19655.250000,35074.898438,70313.796875,848.070269,225.794472,505.476386,8671.604011,15328.919235,19048.765720,6131.750000,1047.300781,5688.304688,NaN,-0.626225,1.138137,1.320339,-0.304359,1.015980,0.630699,0.749273,1.061797,0.441185,0.337143,0.300019,0.311965,0.029859,0.080899,RRLyrae
35,604062754768158861,39.972151,-34.767389,22,9.0,13.0,42.532613,62.028849,76570.100,30979.300,6380.845052,3872.413330,Unlabeled,112648.500000,683248.500000,339278.875000,341.066498,907.497925,574.742615,18.772093,16.809809,17.564342,18.713680,16.713972,17.305283,18.808716,17.028778,17.685646,18.721525,16.719059,17.312044,18.806939,16.987944,17.676164,18.781425,16.743050,17.649525,111551.000000,729091.000000,316366.000000,230.469963,161.546587,

Query the gaia for every ra/dec pair and take gaia's period (if it exists)

In [41]:
Gaia.ROW_LIMIT = -1
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

for index, row in df.iterrows():
    query_gaia_singleObject = f'''SELECT TOP 1 r.source_id, r.pf,
            g.ra, g.dec
            FROM gaiadr3.vari_rrlyrae AS r
            JOIN gaiadr3.gaia_source AS g
            ON r.source_id = g.source_id
            WHERE (
            CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', {row['ra']}, {row['dec']}, 0.0001)) = 1
            )
            '''

    job_gaia_singleObject = Gaia.launch_job_async(query_gaia_singleObject)
    results_gaia_singleObject = job_gaia_singleObject.get_results()
    df_gaia_single_Object = results_gaia_singleObject.to_pandas()
    df_gaia_single_Object
    if not df_gaia_single_Object.empty:
        df.at[index, 'Period'] = df_gaia_single_Object.iloc[0]['pf']

INFO: Query finished. [astroquery.utils.tap.core]


INFO: Query finished. [astroquery.utils.tap.core]


INFO: Query finished. [astroquery.utils.tap.core]


INFO: Query finished. [astroquery.utils.tap.core]


INFO: Query finished. [astroquery.utils.tap.core]


...

INFO: Query finished. [astroquery.utils.tap.core]


INFO: Query finished. [astroquery.utils.tap.core]


INFO: Query finished. [astroquery.utils.tap.core]


INFO: Query finished. [astroquery.utils.tap.core]


INFO: Query finished. [astroquery.utils.tap.core]


In [30]:
pd.reset_option('display.max_rows')

In [31]:
df

,diaObjectId,ra,dec,nDiaSources,i_psfFluxNdata,r_psfFluxNdata,r_psfFluxStetsonJ,i_psfFluxStetsonJ,r_psfFluxChi2,i_psfFluxChi2,r_psfFluxChi2_red,i_psfFluxChi2_red,Class,g_psfFluxMean,i_psfFluxMean,r_psfFluxMean,g_psfFluxErr,i_psfFluxErr,r_psfFluxErr,g_weighted_mean_mag,i_weighted_mean_mag,r_weighted_mean_mag,g_fluxMagMin,i_fluxMagMin,r_fluxMagMin,g_fluxMagMax,i_fluxMagMax,r_fluxMagMax,g_fluxMagPerc05,i_fluxMagPerc05,r_fluxMagPerc05,g_fluxMagPerc95,i_fluxMagPerc95,r_fluxMagPerc95,g_fluxMagMedian,i_fluxMagMedian,r_fluxMagMedian,g_psfFluxMedian,i_psfFluxMedian,r_psfFluxMedian,g_fluxMagMaxSlope,i_fluxMagMaxSlope,r_fluxMagMaxSlope,g_psfFluxSigma,i_psfFluxSigma,r_psfFluxSigma,g_psfFluxMAD,i_psfFluxMAD,r_psfFluxMAD,g_fluxMagSkew,i_fluxMagSkew,r_fluxMagSkew,g_r,r_i,g_i,g_amp,i_amp,r_amp,g_psfFluxSigma_norm,i_psfFluxSigma_norm,r_psfFluxSigma_norm,g_psfFluxMAD_norm,i_psfFluxMAD_norm,r_psfFluxMAD_norm,Predictions,Period
5,604062686048682036,40.152644,-34.701763,19,8.0,11.0,10.587838,7.663866,1783.380,545.885,178.337927,77.983564,RRLyrae,7041.962891,12642.517578,11965.458008,146.826752,231.189667,163.669525,21.782701,21.072393,21.177643,21.728767,20.961226,20.910250,21.823505,21.439362,21.506714,21.733661,20.970693,20.928303,21.821796,21.408802,21.499733,21.786751,21.017780,21.174442,7005.150146,14219.599609,12309.000000,228.855621,242.740452,215.536541,288.352859,2270.292723,1820.218782,199.290283,760.300781,584.299805,-0.391034,0.327232,0.375792,0.605059,0.105249,0.710308,0.088134,0.438109,0.571430,0.040948,0.179576,0.152123,0.028449,0.053469,0.047469,RRLyrae,NaN
6,604062686048682065,40.058354,-34.669641,17,10.0,7.0,13.782720,7.641371,1509.970,743.383,251.660889,82.598165,Unlabeled,13385.049805,19409.099609,20331.078125,158.539001,244.567734,175.794739,21.083750,20.685913,20.618013,21.002224,20.558735,20.478302,21.177109,20.863914,20.963402,21.006704,20.568144,20.493887,21.170298,20.799638,20.874889,21.081898,20.687975,20.602322,13418.300293,19266.800781,20848.300781,351.681086,350.802619,343.919391,1010.573529,1540.891892,2310.213759,810.799805,822.800781,1036.000000,0.144058,0.464276,1.167591,0.465736,-0.067900,0.397837,0.163594,0.231494,0.381002,0.075500,0.079390,0.113630,0.060425,0.042706,0.049692,RRLyrae,NaN
7,604062686048682098,40.120221,-34.784352,16,4.0,12.0,4.581863,3.748847,731.713,124.485,66.519376,41.495008,Unlabeled,7702.240234,11517.312500,10823.839844,152.697998,224.611160,161.242783,21.683405,21.210251,21.301123,21.682003,21.115099,21.047628,21.684441,21.409576,21.452473,21.682196,21.129737,21.070629,21.684389,21.400403,21.451560,21.683929,21.180405,21.356697,7698.890137,12241.599609,10406.950195,4.653597,94.439072,155.460180,9.118822,1267.269619,1199.016929,3.620117,758.900391,682.250488,-1.429044,0.337751,-1.269174,0.382282,0.090872,0.473154,0.002194,0.270666,0.380931,0.001184,0.110032,0.110776,0.000470,0.061994,0.065557,RRLyrae,NaN
8,604062686048682133,40.093003,-34.629648,16,4.0,12.0,4.733237,8.266910,461.790,292.839,41.980924,97.613007,Unlabeled,50388.800781,181444.265625,129995.437500,227.648331,488.071991,363.965942,19.650953,18.253801,18.620802,19.550991,18.154505,18.164358,19.831341,18.295330,18.724960,19.553089,18.194359,18.551682,19.805404,18.294205,18.686539,19.571974,18.265263,18.642912,53853.000000,179430.000000,126717.000000,628.988822,216.408381,777.134497,6930.224528,6865.561071,15224.258336,1050.898438,4539.000000,2606.000000,1.696935,-1.539695,-3.875892,1.030151,0.367001,1.397152,0.252315,0.099846,0.134857,0.137535,0.037838,0.117114,0.019514,0.025297,0.020566,RRLyrae,NaN
20,604062754768158762,39.946244,-34.640705,28,11.0,17.0,44.931238,33.638724,94982.700,20606.500,5936.421387,2060.653516,Unlabeled,19655.250000,45467.070312,63491.824219,165.864990,304.264618,265.667542,20.645439,19.629459,19.325100,20.371498,19.352684,18.979301,21.072275,20.151840,20.224380,20.406537,19.356421,19.096092,21.037236,20.105694,20.157889,20.721886,20.037510,19.282398,19655.250000,35074.898438,70313.796875,848.070269,225.7944

In [104]:
# df = df.dropna(subset=['Period'])

In [32]:
df['Period'].isna().sum()

np.int64(184)

In [ ]:
Export to csv

In [34]:
# df.to_csv('Light_Curve_Stars.csv')
# df.to_csv('predictions_with_periods.csv', index=False)
# df.to_csv('predictions_with_periods_rrl.csv', index=False)
df.to_csv('predictions_with_periods_rrl_candidates.csv', index=False)